# Transformer Architecture Comparison. BEA 2026 Shared Task

**Team Data Asgardians. Notebook 3 of 4 in the reproducibility pipeline.**

A compact notebook for comparing larger transformer models on vocabulary difficulty prediction. Training requires a GPU; the experiments reported in the paper were run on Google Colab Pro (NVIDIA T4 and A100). The official task data must be available by cloning `https://github.com/britishcouncil/bea2026st`.

Two scenarios are covered:
- **Closed**: one model per L1 (`es`, `de`, `cn`)
- **Open**: a single joint model trained on all L1s

Main output: a comparative table of `RMSE` and `Pearson` per model and scenario.

In [ ]:
# Si falta alguna libreria, descomenta:
# !pip install -q transformers datasets accelerate evaluate scipy seaborn

import gc
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ============================
# Configuracion editable
# ============================
SEED = 42
set_seed(SEED)
np.random.seed(SEED)

# Ajusta esta ruta a tu carpeta bea2026st
BEA_DIR = Path('./bea2026st')
DATA_DIR = BEA_DIR / 'data'

EXPERIMENT_DIR = Path('./experiments_transformers_compare')
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'GLMM_score'
L1S = ['es', 'de', 'cn']
INPUT_COLS = ['L1_source_word', 'L1_context', 'en_target_clue', 'en_target_word']

# Modelos para comparar. Puedes agregar/quitar libremente.
MODEL_CANDIDATES = [
    'xlm-roberta-base',
    'xlm-roberta-large',
    'microsoft/mdeberta-v3-base',
    'microsoft/mdeberta-v3-large'
]

# Para pruebas rapidas
FAST_MODE = False
MAX_SAMPLES_PER_L1 = 1000

# Ajustes de entrenamiento (pensados para modelos grandes)
MAX_LENGTH = 192
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.1
BATCH_SIZE = 4
GRAD_ACC = 4
EARLY_STOPPING_PATIENCE = 2

USE_FP16 = torch.cuda.is_available()
print('Experiment dir:', EXPERIMENT_DIR.resolve())

In [ ]:
def load_split(split_name: str, l1s=L1S) -> pd.DataFrame:
    frames = []
    for l1 in l1s:
        csv_path = DATA_DIR / split_name / l1 / f'kvl_shared_task_{l1}_{split_name}.csv'
        if not csv_path.exists():
            print(f'No encontrado: {csv_path}')
            continue
        df = pd.read_csv(csv_path)
        df['split'] = split_name
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def build_input_text(row: pd.Series) -> str:
    return ' '.join(str(row[c]) for c in INPUT_COLS)

train_df = load_split('train')
dev_df = load_split('dev')
test_df = load_split('test')

if train_df.empty or dev_df.empty:
    raise ValueError('No se pudo cargar train/dev. Revisa BEA_DIR y DATA_DIR.')

for df in [train_df, dev_df, test_df]:
    if not df.empty:
        df['text'] = df.apply(build_input_text, axis=1)

if FAST_MODE:
    train_chunks = []
    dev_chunks = []
    for l1 in L1S:
        t = train_df[train_df['L1'] == l1].sample(n=min(MAX_SAMPLES_PER_L1, (train_df['L1'] == l1).sum()), random_state=SEED)
        d = dev_df[dev_df['L1'] == l1].sample(n=min(MAX_SAMPLES_PER_L1, (dev_df['L1'] == l1).sum()), random_state=SEED)
        train_chunks.append(t)
        dev_chunks.append(d)
    train_df = pd.concat(train_chunks, ignore_index=True)
    dev_df = pd.concat(dev_chunks, ignore_index=True)

print('Train:', train_df.shape, train_df['L1'].value_counts().to_dict())
print('Dev  :', dev_df.shape, dev_df['L1'].value_counts().to_dict())
if not test_df.empty:
    print('Test :', test_df.shape, test_df['L1'].value_counts().to_dict())

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_pearson(y_true, y_pred):
    if len(y_true) < 3:
        return 0.0
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0
    return float(pearsonr(y_true, y_pred)[0])

def metrics_from_predictions(y_true, y_pred):
    return {
        'rmse': rmse(y_true, y_pred),
        'pearson': safe_pearson(y_true, y_pred),
    }

def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    cols = ['item_id', 'L1', 'text', TARGET_COL]
    data = df[cols].copy()
    data = data.rename(columns={TARGET_COL: 'labels'})
    data['labels'] = data['labels'].astype(np.float32)
    return Dataset.from_pandas(data, preserve_index=False)

def tokenize_dataset(ds: Dataset, tokenizer, max_length=MAX_LENGTH) -> Dataset:
    def _tok(batch):
        return tokenizer(batch['text'], truncation=True, max_length=max_length)
    tokenized = ds.map(_tok, batched=True)
    tokenized = tokenized.remove_columns(['text'])
    tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized

In [ ]:
from transformers import EarlyStoppingCallback

def train_and_eval_once(model_name: str, train_part: pd.DataFrame, dev_part: pd.DataFrame, run_tag: str):
    run_dir = EXPERIMENT_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    train_ds = tokenize_dataset(make_hf_dataset(train_part), tokenizer)
    dev_ds = tokenize_dataset(make_hf_dataset(dev_part), tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

    # Reduce memoria en modelos grandes
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

    args = TrainingArguments(
        output_dir=str(run_dir),
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='rmse',
        greater_is_better=False,
        save_total_limit=1,
        fp16=USE_FP16,
        report_to='none',
        remove_unused_columns=False,
    )

    def compute_metrics(eval_pred):
        pred = eval_pred.predictions
        if isinstance(pred, tuple):
            pred = pred[0]
        pred = np.squeeze(pred)
        y = eval_pred.label_ids
        return metrics_from_predictions(y, pred)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer.train()
    pred_out = trainer.predict(dev_ds)
    y_true = pred_out.label_ids
    y_pred = np.squeeze(pred_out.predictions)

    metrics = metrics_from_predictions(y_true, y_pred)
    dev_ids = dev_part['item_id'].to_numpy()

    pred_df = pd.DataFrame({
        'item_id': dev_ids,
        'prediction': y_pred,
    })
    pred_df.to_csv(run_dir / 'dev_predictions.csv', index=False)

    del trainer, model, tokenizer, train_ds, dev_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

In [ ]:
# ============================
# Ejecutar comparativas
# ============================
all_rows = []

# 1) Closed: un modelo por L1
for model_name in MODEL_CANDIDATES:
    model_slug = model_name.replace('/', '__')
    for l1 in L1S:
        tr = train_df[train_df['L1'] == l1].copy()
        dv = dev_df[dev_df['L1'] == l1].copy()
        if len(tr) == 0 or len(dv) == 0:
            continue

        run_tag = f'closed__{model_slug}__{l1}'
        print(f'\n[Closed] {model_name} | L1={l1} | train={len(tr)} dev={len(dv)}')
        metrics = train_and_eval_once(model_name, tr, dv, run_tag=run_tag)

        all_rows.append({
            'scenario': 'closed',
            'model': model_name,
            'l1': l1,
            **metrics,
        })

# 2) Open: un modelo conjunto
for model_name in MODEL_CANDIDATES:
    model_slug = model_name.replace('/', '__')
    run_tag = f'open__{model_slug}__all'

    print(f'\n[Open] {model_name} | train={len(train_df)} dev={len(dev_df)}')
    _ = train_and_eval_once(model_name, train_df, dev_df, run_tag=run_tag)

    pred_path = EXPERIMENT_DIR / run_tag / 'dev_predictions.csv'
    pred_df = pd.read_csv(pred_path)
    merged = pred_df.merge(dev_df[['item_id', 'L1', TARGET_COL]], on='item_id', how='inner')

    for l1 in L1S:
        sub = merged[merged['L1'] == l1]
        if len(sub) == 0:
            continue
        m = metrics_from_predictions(sub[TARGET_COL].to_numpy(), sub['prediction'].to_numpy())
        all_rows.append({
            'scenario': 'open',
            'model': model_name,
            'l1': l1,
            **m,
        })

results_df = pd.DataFrame(all_rows)
results_df.to_csv(EXPERIMENT_DIR / 'results_comparative.csv', index=False)
print('\nGuardado:', EXPERIMENT_DIR / 'results_comparative.csv')
display(results_df.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if results_df.empty:
    raise ValueError('No hay resultados para visualizar.')

results_sorted = results_df.sort_values(['scenario', 'l1', 'rmse']).reset_index(drop=True)
display(results_sorted)

summary = (
    results_df.groupby(['scenario', 'model'], as_index=False)
    .agg(mean_rmse=('rmse', 'mean'), mean_pearson=('pearson', 'mean'))
    .sort_values('mean_rmse')
)
print('Resumen por escenario/modelo (promedio sobre L1):')
display(summary)

plt.figure(figsize=(14, 5))
sns.barplot(data=results_df, x='l1', y='rmse', hue='model', palette='viridis')
plt.title('RMSE por L1 y modelo (mezcla closed/open)')
plt.grid(axis='y', alpha=0.2)
plt.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
sns.barplot(data=results_df, x='l1', y='pearson', hue='model', palette='magma')
plt.title('Pearson por L1 y modelo (mezcla closed/open)')
plt.grid(axis='y', alpha=0.2)
plt.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Opcional: entrenar un modelo open y generar predicciones para test.

if test_df.empty:

    print('No hay test_df cargado; salto este bloque.')

else:

    def train_open_and_predict_test(model_name: str, out_name: str):

        run_tag = f'test_open__{model_name.replace("/", "__")}'

        run_dir = EXPERIMENT_DIR / run_tag

        run_dir.mkdir(parents=True, exist_ok=True)



        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        train_ds = tokenize_dataset(make_hf_dataset(train_df), tokenizer)

        dev_ds = tokenize_dataset(make_hf_dataset(dev_df), tokenizer)



        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

        if hasattr(model, 'gradient_checkpointing_enable'):

            model.gradient_checkpointing_enable()



        args = TrainingArguments(

            output_dir=str(run_dir),

            learning_rate=LR,

            per_device_train_batch_size=BATCH_SIZE,

            per_device_eval_batch_size=BATCH_SIZE * 2,

            gradient_accumulation_steps=GRAD_ACC,

            num_train_epochs=EPOCHS,

            weight_decay=WEIGHT_DECAY,

            evaluation_strategy='epoch',

            save_strategy='epoch',

            load_best_model_at_end=True,

            metric_for_best_model='rmse',

            greater_is_better=False,

            save_total_limit=1,

            fp16=USE_FP16,

            report_to='none',

            remove_unused_columns=False,

        )



        def compute_metrics(eval_pred):

            pred = eval_pred.predictions

            if isinstance(pred, tuple):

                pred = pred[0]

            pred = np.squeeze(pred)

            y = eval_pred.label_ids

            return metrics_from_predictions(y, pred)



        trainer = Trainer(

            model=model,

            args=args,

            train_dataset=train_ds,

            eval_dataset=dev_ds,

            tokenizer=tokenizer,

            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),

            compute_metrics=compute_metrics,

            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],

        )



        trainer.train()



        test_ds = Dataset.from_pandas(test_df[['item_id', 'L1', 'text']].copy(), preserve_index=False)

        test_tok = test_ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH), batched=True)

        test_tok.set_format(type='torch', columns=['input_ids', 'attention_mask'])



        pred_out = trainer.predict(test_tok)

        test_pred = np.squeeze(pred_out.predictions)



        out_df = pd.DataFrame({'item_id': test_df['item_id'].to_numpy(), 'prediction': test_pred})

        out_path = EXPERIMENT_DIR / out_name

        out_df.to_csv(out_path, index=False)

        print('Predicciones test guardadas en:', out_path)



        del trainer, model, tokenizer, train_ds, dev_ds, test_ds, test_tok

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()



    # Usa el mejor modelo por RMSE medio si ya existe summary; si no, usa xlm-roberta-large.

    model_for_test = summary.iloc[0]['model'] if 'summary' in globals() and len(summary) > 0 else 'xlm-roberta-large'

    train_open_and_predict_test(model_for_test, 'test_predictions_open_best.csv')


## Notes for experimentation

1. Start with `FAST_MODE = True` to validate the pipeline.
2. Then switch to `FAST_MODE = False` and adjust `EPOCHS`.
3. If VRAM runs out: lower `BATCH_SIZE`, raise `GRAD_ACC`, or lower `MAX_LENGTH` to 160 or 128.
4. To evaluate new models, simply add them to `MODEL_CANDIDATES`.
5. Final results are written to `results_comparative.csv`.